In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import numpy as np
import pandas as pd
from keras.models import Model, Sequential
from keras.layers import Input, Dense, Conv1D, Flatten, MaxPooling1D, Concatenate, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.metrics import confusion_matrix

In [ ]:
%matplotlib inline

In [ ]:
value_df = 10000
star_check = pd.read_csv("./Datasets/final_dataset.csv")
star_check = star_check.drop(['tid'],axis=1)
star_check_y = star_check[['confirmed_planet']]
star_check = star_check.reset_index().drop('index',axis=1)
star_check = star_check.apply(lambda row: row.fillna(0), axis=1)

In [ ]:
avg_val = int(star_check.value_counts(star_check['confirmed_planet']).mean())
balanced_star_check = pd.DataFrame()
class_counts = star_check['confirmed_planet'].value_counts()
for i in range(0,2):
    if class_counts.loc[class_counts.index == i].iloc[0] > avg_val:
        balanced_star_check = pd.concat([balanced_star_check, star_check[star_check['confirmed_planet'] == i].sample(avg_val)])
    else:
        balanced_star_check = pd.concat([balanced_star_check, star_check[star_check['confirmed_planet'] == i].sample(avg_val, replace=True)])
star_check = balanced_star_check.copy()

In [ ]:
scaler = MinMaxScaler()
star_check[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']] = scaler.fit_transform(star_check[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(star_check.drop('confirmed_planet',axis=1),star_check[['confirmed_planet']], test_size=0.1, random_state=42)

In [ ]:
X_train_flux = X_train.drop(['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx'],axis=1)
X_train_params = X_train[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]
X_test_flux = X_test.drop(['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx'],axis=1)
X_test_params = X_test[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]

In [ ]:
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(10000,1)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dropout(0.75),
    Flatten(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# model.fit(X_train_flux, y_train, epochs=30, batch_size=32, validation_data=(X_test_flux, y_test))

In [ ]:
for i in range(1):
    print(i)

In [ ]:
from astroquery.mast import Catalogs
Catalogs.query_criteria(catalog='Tic', ID=139086171)

In [ ]:
import csv

with open('./Datasets/exoplanet_star_normal_flux.csv', 'r') as f:
    data = f.readlines()

In [ ]:
data_sizes = [ len(i)-1 for i in data[1:]]

In [ ]:
max(data_sizes)

In [ ]:
min(data_sizes)

In [ ]:
max(data_sizes)/min(data_sizes)

In [ ]:
import math

num1 = 1574525
num2 = 11159

# Find the GCD of num1 and num2
gcd = math.gcd(num1, num2)

# Divide num1 and num2 by the GCD to simplify the proportion
simplified_num1 = num1 // gcd
simplified_num2 = num2 // gcd

print(f"Simplified proportion is {simplified_num1}:{simplified_num2}")

In [ ]:
import astropy.io.fits as fits
import lightkurve as lk
from astropy.time import Time
import numpy as np

# Read the FITS file
hdulist = fits.open('test_fits.fits')
data = hdulist[1].data

# Extract relevant data
time = data['TIME']
flux = data['PDCSAP_FLUX']
error = data['PDCSAP_FLUX_ERR']

finite_indices = np.isfinite(time)
clean_time = time[finite_indices]

lc = lk.LightCurve(time=Time(clean_time, format='jd'), flux=flux[finite_indices], flux_err=error[finite_indices])

# Analyze and visualize the light curve
lc.plot()


In [ ]:
def process_file(file):
    try:
        with fits.open(file) as hdulist:
            time = hdulist[1].data['TIME'].astype(float)
            flux = hdulist[1].data['PDCSAP_FLUX'].astype(float)
            error = hdulist[1].data['PDCSAP_FLUX_ERR'].astype(float)

            # Only keep finite values for processing
            finite_mask = np.isfinite(time) & np.isfinite(flux) & np.isfinite(error)
            time, flux, error = time[finite_mask], flux[finite_mask], error[finite_mask]

            lc = lk.LightCurve(time=Time(time, format='jd'), flux=flux, flux_err=error)

            # Simple example of folding and plotting
            period = 10  # This should be determined via analysis such as BLS
            folded_lc = lc.fold(period=period)
            plt.figure()
            plt.plot(folded_lc.time.value, folded_lc.flux, 'k.')
            plt.xlabel('Phase')
            plt.ylabel('Flux')
            plt.title('Folded Light Curve')
            plt.savefig(general_plot_path)
            plt.close()
        
            stitched_lc = lc.stitch() 

            min_period = 1
            max_period = 1000
            num_periods = 10000
            period_time = np.logspace(np.log10(min_period), np.log10(max_period), num_periods)
            bls_periodogram = stitched_lc.to_periodogram(method='bls', period=period_time)
            planet_period = bls_periodogram.period_at_max_power
            planet_t0 = bls_periodogram.transit_time_at_max_power
            folded_light_curve = stitched_lc.fold(period=planet_period, epoch_time=planet_t0)
            print("trial")
            flatten_lc, trend_lc = flatten(folded_light_curve.time.value, folded_light_curve.flux, method='biweight', return_trend=True)
            print("trial2")
            light = pd.DataFrame({'Time':folded_light_curve.time.value,'Flux':flatten_lc}).dropna()
            print("okay?")
            flux_series = pd.Series([i[0] for i in light[['Flux']].to_numpy()], index=[i[0] for i in light[['Time']].to_numpy()])
            print("okay?")
            decompose_result_mult = seasonal_decompose(flux_series, model="additive", period=int(planet_period.value))
            print("okay?")
            trend = TimeSeriesResampler(sz=10000).fit_transform(np.array(decompose_result_mult.trend))[0]
            
            status = 0
            model = input_flux = Input(shape=(10000,1))
            x = Conv1D(filters=64, kernel_size=3, activation='relu')(input_flux)
            x = MaxPooling1D(pool_size=2)(x)
            x = Conv1D(filters=64, kernel_size=5, activation='relu')(x)
            x = MaxPooling1D(pool_size=2)(x)
            x = Conv1D(filters=64, kernel_size=5, activation='relu')(x)
            x = MaxPooling1D(pool_size=2)(x)
            x = Flatten()(x)
            x = Dropout(0.75)(x)
            model_flux = Model(inputs=input_flux, outputs=x)

            input_params = Input(shape=(11,))
            y = Dense(128, activation='relu')(input_params)
            model_params = Model(inputs=input_params, outputs=y)

            combined = Concatenate()([model_flux.output, model_params.output])
            z = Dropout(0.5)(combined)
            z = Dense(128, activation='relu')(z)
            final_output = Dense(1, activation='sigmoid')(z)

            model = Model(inputs=[model_flux.input, model_params.input], outputs=final_output)
            
            model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
            
            model.load_weights('../../Models/model_weights_v5.h5')
            
            catalog_data = Catalogs.query_criteria(catalog="Tic", ID=int(identifier[4:]))[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']].to_pandas().fillna(0)
            
            scaler = MinMaxScaler()
            catalog_data = scaler.fit_transform(catalog_data)
        
            status_val = model.predict([trend.reshape(1, 10000, 1), catalog_data.reshape(1, -1)], verbose=0)
            if status_val == 1:
                status = "Potential Exoplanet Candidate"
            else:
                status = "No Exoplanet Transit Signals Detected"
            
            plt.figure(figsize=(9.10, 2.15))
            plt.plot(range(10000), trend, color="#FF4C29")
            plt.gcf().set_facecolor('#202123')  # Set background color of the figure
            plt.grid(False)
            plt.axis('off')
            plt.savefig(trend_plot_path)
            plt.close()

            return {
            'general_plot_url': f'/api/images/custom_general_plot.png',
            'trend_plot_url': f'/api/images/custom_trend_plot.png',
            'status': status
            }
    except Exception as e:
        return {"error": str(e)}, 500

In [ ]:
process_file('test_fits.fits')

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
# Define the model
model = Sequential()
model.add(Conv1D(32, 3, activation='relu', input_shape=(10000, 1)))
model.add(MaxPooling1D(2))
model.add(Conv1D(64, 3, activation='relu'))
model.add(MaxPooling1D(2))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()
    


In [ ]:
model.fit(X_train_flux, y_train, epochs=5, batch_size=32, validation_data=(X_test_flux, y_test))

In [ ]:
model.save('model_for_ui.h5', overwrite=True)